In [1]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()
print("✅ Memory cleared!")

✅ Memory cleared!


In [2]:
# PROJECT 2 - PART 1: SUPERVISED FINE-TUNING


!pip install -U --upgrade-strategy only-if-needed trl bitsandbytes accelerate transformers peft datasets torchao -q

import pandas as pd
import torch
import os
from sklearn.model_selection import train_test_split
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, DPOConfig, DPOTrainer
from transformers import pipeline, GenerationConfig
from transformers import set_seed
from huggingface_hub import notebook_login
from datasets import Dataset

print("✅ Libraries installed and imported!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.0 MB/s eta 0:00:00


✅ Libraries installed and imported!


In [3]:
notebook_login()

In [5]:
df = pd.read_csv("/content/project2_dataset.csv")
print(f"Loaded {len(df)} samples")
print(f" Columns: {df.columns.tolist()}")

Loaded 2000 samples
 Columns: ['index', 'sample', 'ground_truth', 'cot', 'reject', 'new_chosen']


In [7]:
STUDENT_ID = 2015973

training_data, testing_data = train_test_split(
    df,
    test_size=20,
    random_state=STUDENT_ID,
    shuffle=True
)

training_data = training_data.reset_index(drop=True)
testing_data = testing_data.reset_index(drop=True)

testing_data.to_csv(f"{STUDENT_ID}_testing_data.csv", index=False)


print(testing_data[['sample', 'ground_truth']].head())

                                              sample  \
0     paperclip, carbon atom, mouse, toaster, saturn   
1        lion, galaxy, coin, skyscraper, carbon atom   
2  skyscraper, continent, grain of sand, city, horse   
3  smartphone, football, water molecule, carbon a...   
4       milky way, state, laptop, grain of salt, dog   

                                        ground_truth  
0  saturn -> toaster -> mouse -> paperclip -> car...  
1  galaxy -> skyscraper -> lion -> coin -> carbon...  
2  continent -> city -> skyscraper -> horse -> gr...  
3  skyscraper -> football -> smartphone -> water ...  
4  milky way -> state -> dog -> laptop -> grain o...  


In [8]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"\n⏳ Loading {MODEL_NAME}...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

print("✅ Base model loaded!")


⏳ Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

...

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

✅ Base model loaded!


In [9]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

print(" LoRA config ready!")

 LoRA config ready!


In [10]:
def format_sft(row):
    prompt = f"""Sort the following objects from BIGGEST to SMALLEST.

Objects: {row['sample']}

Please provide your reasoning step by step, then output the sorted list.

OUTPUT:"""

    response = row['cot']

    return {"prompt": prompt, "response": response}

sft_data = training_data.apply(format_sft, axis=1)
sft_list = sft_data.tolist()

print(f"SFT data formatted: {len(sft_list)} samples")
print("\n First SFT sample:")
print(f"Prompt: {sft_list[0]['prompt'][:100]}...")
print(f"Response: {sft_list[0]['response'][:100]}...")

SFT data formatted: 1980 samples

 First SFT sample:
Prompt: Sort the following objects from BIGGEST to SMALLEST.

Objects: peanut, oort cloud, eagle, ping pong ...
Response: REASONING:  
1. **Identify the largest object:** The Oort Cloud is a spherical shell of icy bodies s...


In [11]:
# Split data into phases
phase_size = 500
phases = []
for i in range(0, len(sft_list), phase_size):
    phases.append(sft_list[i:i+phase_size])

print(f"Total phases: {len(phases)}")
for i, phase in enumerate(phases):
    print(f"Phase {i+1}: {len(phase)} samples")

Total phases: 4
Phase 1: 500 samples
Phase 2: 500 samples
Phase 3: 500 samples
Phase 4: 480 samples


In [12]:
import os
import zipfile
from google.colab import files

def download_adapter(phase_num):
    if os.path.exists("./sft_adapter"):
        with zipfile.ZipFile(f"sft_adapter_phase{phase_num}.zip", 'w') as zipf:
            for root, dirs, files in os.walk("./sft_adapter"):
                for file in files:
                    zipf.write(os.path.join(root, file),
                              os.path.relpath(os.path.join(root, file), "./sft_adapter"))
        files.download(f"sft_adapter_phase{phase_num}.zip")
        print(f" Downloaded: sft_adapter_phase{phase_num}.zip")
    else:
        print(" sft_adapter folder not found")

print(" Download function ready!")

 Download function ready!


In [17]:
from transformers import TrainingArguments
from trl import SFTTrainer
from datasets import Dataset

print("="*50)
print("PHASE 1: Training on samples 0-500")
print("="*50)

# Convert phase 0 to Dataset and rename column to 'completion'
phase0_list = phases[0]
for item in phase0_list:
    if 'response' in item:
        item['completion'] = item.pop('response')

phase0_dataset = Dataset.from_list(phase0_list)

training_args = TrainingArguments(
    output_dir="./sft_output",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    save_strategy="no",
    logging_steps=10,
    optim="paged_adamw_8bit",
    report_to="none",
    max_grad_norm=0.3,
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=base_model,
    args=training_args,
    train_dataset=phase0_dataset,
    peft_config=lora_config,
)

print("⏳ Phase 1 Training (500 samples, 2 epochs)...")
trainer.train()
print("✅ Phase 1 complete!")

trainer.save_model("./sft_adapter")
print("✅ SFT adapter saved!")

# Download Phase 1 adapter
download_adapter(1)

PHASE 1: Training on samples 0-500


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

...

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


⏳ Phase 1 Training (500 samples, 2 epochs)...


Step,Training Loss
10,1.443647
20,1.160606
30,1.020299
40,0.986471
50,0.965659
60,0.944568
70,0.873160
80,0.853249
90,0.847166
100,0.831886


✅ Phase 1 complete!
✅ SFT adapter saved!


AttributeError: 'list' object has no attribute 'download'

In [19]:
import os
import zipfile
from google.colab import files

def download_adapter(phase_num):
    if os.path.exists("./sft_adapter"):
        zip_path = f"sft_adapter_phase{phase_num}.zip"
        with zipfile.ZipFile(zip_path, 'w') as zipf:
            for root, dirs, files_list in os.walk("./sft_adapter"):
                for file in files_list:
                    zipf.write(os.path.join(root, file),
                              os.path.relpath(os.path.join(root, file), "./sft_adapter"))
        # Download using google.colab.files
        files.download(zip_path)
        print(f" Downloaded: {zip_path}")
    else:
        print("❌ sft_adapter folder not found")

print(" Download function ready!")

 Download function ready!


In [20]:
# Download Phase 1 adapter
download_adapter(1)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Downloaded: sft_adapter_phase1.zip


In [23]:
print("="*50)
print("PHASE 2: Training on samples 500-1000")
print("="*50)

# Convert phase 1 to Dataset and rename column to 'completion'
phase1_list = phases[1]
for item in phase1_list:
    if 'response' in item:
        item['completion'] = item.pop('response')

phase1_dataset = Dataset.from_list(phase1_list)

training_args = TrainingArguments(
    output_dir="./sft_output",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    save_strategy="no",
    logging_steps=10,
    optim="paged_adamw_8bit",
    report_to="none",
    max_grad_norm=0.3,
    gradient_checkpointing=True,
)

# Use base_model (not PeftModel) and let SFTTrainer apply LoRA
trainer = SFTTrainer(
    model=base_model,
    args=training_args,
    train_dataset=phase1_dataset,
    peft_config=lora_config,
)

print("⏳ Phase 2 Training (500 samples, 2 epochs)...")
trainer.train()
print(" Phase 2 complete!")

trainer.save_model("./sft_adapter")
print(" SFT adapter saved! (Phase 1 + Phase 2 combined)")

download_adapter(2)

PHASE 2: Training on samples 500-1000


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

...

Step,Training Loss
10,1.419520
20,1.170143
30,1.047002
40,0.984053
50,0.954849
60,0.930476
70,0.863041
80,0.857594
90,0.801564
100,0.853414


 Phase 2 complete!
 SFT adapter saved! (Phase 1 + Phase 2 combined)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Downloaded: sft_adapter_phase2.zip


In [24]:
print("="*50)
print("PHASE 3: Training on remaining samples 1000-1980")
print("="*50)

# Combine phase 3 and 4
remaining_list = phases[2] + phases[3]


for item in remaining_list:
    if 'response' in item:
        item['completion'] = item.pop('response')

remaining_dataset = Dataset.from_list(remaining_list)

print(f"✅ Using {len(remaining_list)} samples (1000-1980)")

training_args = TrainingArguments(
    output_dir="./sft_output",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    save_strategy="no",
    logging_steps=10,
    optim="paged_adamw_8bit",
    report_to="none",
    max_grad_norm=0.3,
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=base_model,
    args=training_args,
    train_dataset=remaining_dataset,
    peft_config=lora_config,
)

print(" Phase 3 Training (980 samples, 2 epochs)...")
trainer.train()
print(" Phase 3 complete!")

trainer.save_model("./sft_adapter")
print(" SFT adapter saved! (All 1980 samples trained)")

download_adapter(3)

PHASE 3: Training on remaining samples 1000-1980
✅ Using 980 samples (1000-1980)


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/980 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/980 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/980 [00:00<?, ? examples/s]

...

Step,Training Loss
10,1.414354
20,1.124925
30,1.084460
40,0.964416
50,0.932990
60,0.920229
70,0.880870
80,0.898533
90,0.852503
100,0.848707


 Phase 3 complete!
 SFT adapter saved! (All 1980 samples trained)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Downloaded: sft_adapter_phase3.zip


In [25]:
from peft import PeftModel
from transformers import pipeline, GenerationConfig
from transformers import set_seed

# Load SFT adapter
sft_model = PeftModel.from_pretrained(base_model, "./sft_adapter")
sft_model = sft_model.merge_and_unload()
sft_model.eval()

generator = pipeline(
    task="text-generation",
    model=sft_model,
    tokenizer=tokenizer,
    clean_up_tokenization_spaces=False,
)

def generate_response(prompt, generator, seed=42):
    gen_config = GenerationConfig(
        temperature=0.1,
        do_sample=True,
        max_new_tokens=256,
    )
    set_seed(seed)
    messages = [{"role": "user", "content": prompt}]
    outputs = generator(messages, generation_config=gen_config)

    full_output = outputs[0]['generated_text']
    if isinstance(full_output, list):
        for msg in reversed(full_output):
            if msg.get('role') == 'assistant':
                response_text = msg.get('content', '')
                if "OUTPUT:" in response_text:
                    return "OUTPUT: " + response_text.split("OUTPUT:")[-1].strip()
                return response_text.strip()
    if isinstance(full_output, str):
        if "OUTPUT:" in full_output:
            return "OUTPUT: " + full_output.split("OUTPUT:")[-1].strip()
        return full_output.strip()
    return str(full_output)

results = []

for idx, row in testing_data.iterrows():
    prompt = f"""Sort the following objects from BIGGEST to SMALLEST.

Objects: {row['sample']}

Please provide your reasoning step by step, then output the sorted list.

OUTPUT:"""

    output = generate_response(prompt, generator)

    results.append({
        'sample_index': idx,
        'sample': row['sample'],
        'llm_output': output
    })

results_df = pd.DataFrame(results)
results_df.to_csv("part1.csv", index=False)

print(" Part 1 results saved to part1.csv")
print("\n First 3 results:")
print(results_df.head(3))
print("\n PART 1 (SFT) COMPLETE!")

files.download("part1.csv")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

 Part 1 results saved to part1.csv

 First 3 results:
   sample_index                                             sample  \
0             0     paperclip, carbon atom, mouse, toaster, saturn   
1             1        lion, galaxy, coin, skyscraper, carbon atom   
2             2  skyscraper, continent, grain of sand, city, horse   

                                          llm_output  
0  1. **Saturn vs everything else** – Saturn is a...  
1  1. **Galaxy vs. everything else** – A galaxy i...  
2  1. **Identify the largest object** – A contine...  

 PART 1 (SFT) COMPLETE!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>